# Cleaning + One-Hot Encoding
Same cleaning as the other notebook, **plus** one-hot encoding at the end. Saves the encoded data ready for modelling.

### Import tools and load the data

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt

file_path = './credit_risk_dataset.csv'
df = pd.read_csv(file_path)

### First look

In [ ]:
print(df.head())
print('Shape:', df.shape)
print(df.isnull().sum())
print(df.describe())

### Step 1 — Handle missing values
`person_emp_length` → median. `loan_int_rate` → median of its own loan grade.

In [ ]:
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())

df['loan_int_rate'] = df.groupby('loan_grade')['loan_int_rate'].transform(
    lambda s: s.fillna(s.median())
)

print(df.isnull().sum())

### Step 2 — Handle outliers
**(a)** Drop impossible values. **(b)** Cap extreme income with the IQR fence (cap, don't delete).

In [ ]:
# (a) remove impossible values
df = df[df['person_age'] <= 100]
df = df[df['person_emp_length'] <= 60]

# (b) cap extreme income using the IQR fence
Q1 = df['person_income'].quantile(0.25)
Q3 = df['person_income'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR
df['person_income'] = df['person_income'].clip(upper=upper_fence)

df = df.reset_index(drop=True)
print('Rows:', df.shape[0], '| Max income:', df['person_income'].max())

### Step 3 — Create customer_id
Unique ID like `CUST_00001` as the first column.

In [ ]:
df.insert(0, 'customer_id', ['CUST_' + str(i + 1).zfill(5) for i in range(len(df))])
print(df.head())

### Step 4 — One-Hot Encoding
Turn the text categories into 0/1 columns. `customer_id` is kept out of the encoding then put back in front. True/False dummies become clean 0/1.

In [ ]:
categorical_cols = ['person_home_ownership', 'loan_intent',
                    'loan_grade', 'cb_person_default_on_file']

ids = df['customer_id']
df_encoded = pd.get_dummies(df.drop(columns=['customer_id']), columns=categorical_cols)

bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

df_encoded.insert(0, 'customer_id', ids)
print(df_encoded.head())

### Save the encoded file
Same rows, categories now numeric 0/1 columns.

In [ ]:
df_encoded.to_csv('./credit_risk_encoded.csv', index=False)
print('Saved encoded file:', df_encoded.shape)